In [1]:
# notebooks/local_llm_uncertainty.py
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---

# Free local-LLM narrative analysis (no API key, $0)

puremacro's LLM features can run on a model on **your own machine** — Apple
MLX, llama.cpp, or a local Ollama/LM Studio server — instead of a paid API.

**Desktop only:** local inference needs a real engine, so this notebook does
not run a model inside the browser playground. With no engine installed it
falls back to an offline Mock so the notebook still executes; install one with
`pip install "puremacro[local-llm]"` (or run Ollama) to see real inference.

In [2]:
import _nbstyle  # noqa: F401  (grayscale figure style; see notebooks/_nbstyle.py)

from puremacro.narrative.scoring import get_default_backend, score_llm
from puremacro.narrative.indices import get_default_provider, llm_prob_kernel

CORPUS = [
    ("2020-03-15",
     "The government announced a $500 billion infrastructure investment package.",
     "http://example.test/a"),
    ("2020-04-01",
     "Officials warned the outlook is highly uncertain and could shift abruptly.",
     "http://example.test/b"),
]

## 1. Pick the best available local engine
`get_default_backend` / `get_default_provider` auto-select MLX -> llama.cpp ->
Ollama, falling back to a Mock if none is installed (which is what happens in
CI / the browser).

In [3]:
backend = get_default_backend("qwen2.5-3b-instruct")
provider = get_default_provider("qwen2.5-3b-instruct")

[puremacro] No local LLM engine available. Install one with `pip install puremacro[local-llm]` (MLX on Apple Silicon, or llama-cpp-python anywhere), or start Ollama (https://ollama.com) and run `ollama pull qwen2.5:3b`.
[puremacro] using MockBackend (zero events).
[puremacro] No local LLM engine available. Install one with `pip install puremacro[local-llm]` (MLX on Apple Silicon, or llama-cpp-python anywhere), or start Ollama (https://ollama.com) and run `ollama pull qwen2.5:3b`.
[puremacro] using MockProvider.


## 2. Extract narrative fiscal events (free)

In [4]:
events = score_llm(CORPUS, backend=backend, kind="fiscal")
print(f"extracted {len(events)} event(s)")
for ev in events:
    print(ev.date.date(), ev.sign, ev.magnitude, ev.magnitude_unit)

extracted 0 event(s)


## 3. Build a per-document uncertainty index (free)

In [5]:
series = list(llm_prob_kernel(CORPUS, provider=provider,
                              category="economic uncertainty"))
for date, p in series:
    print(date.date(), round(p, 3))

2020-03-15 0.0
2020-04-01 1.0


With a real engine installed, the April "uncertain" document scores higher
than the March "investment" document. Swap models via the `model=` argument
(e.g. `"gemma2-2b"` for Google's Gemma, `"llama3.2-3b"` for Meta's Llama).